In [10]:
# Configuración inicial del entorno
import sys
import os
from pathlib import Path

print("🔄 Configurando entorno...")

# Obtener el directorio raíz del proyecto
notebook_dir = Path().resolve()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    # Buscar el directorio con pyproject.toml
    project_root = notebook_dir
    while project_root != project_root.parent:
        if (project_root / "pyproject.toml").exists():
            break
        project_root = project_root.parent

print(f"📁 Directorio raíz: {project_root}")

# Cambiar al directorio raíz
os.chdir(project_root)
print(f"✓ Directorio de trabajo: {os.getcwd()}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar pyproject.toml
if (project_root / "pyproject.toml").exists():
    print("✓ pyproject.toml encontrado")
else:
    raise FileNotFoundError(f"pyproject.toml no encontrado en {project_root}")

print("✅ Configuración completada\n")


🔄 Configurando entorno...
📁 Directorio raíz: C:\Users\PC RST\Documents\GitHub\ML_ClashRoyale
✓ Directorio de trabajo: C:\Users\PC RST\Documents\GitHub\ML_ClashRoyale
✓ pyproject.toml encontrado
✅ Configuración completada



In [11]:
# 📘 Fase 1: Comprensión del Negocio (CRISP-DM)


# 📘 Fase 1: Comprensión del Negocio (CRISP-DM)

Este notebook documenta la **primera fase de CRISP-DM: Comprensión del Negocio**.  
Se relaciona directamente con el pipeline de *business_understanding*, que incluye:

- Limpieza inicial de datasets de combates (renombrado de primera columna).  
- Análisis de objetivos del negocio.  
- Evaluación de la situación actual.  
- Definición de objetivos de Machine Learning.  
- Generación del plan del proyecto.  
- Creación de un resumen ejecutivo.


## 1. Contexto del Proyecto

- Juego: **Clash Royale**  
- Datos: Registros de combates de distintos días de una temporada.  
- Enfoque del análisis:
  1. Cartas más usadas
  2. Win conditions más frecuentes
  3. Distribución de rarezas en los mazos


## 2. Objetivos del Negocio

En base al análisis inicial del dataset, los objetivos son:

- Identificar qué cartas y combinaciones dominan la temporada.  
- Determinar las condiciones de victoria más frecuentes en distintos días.  
- Analizar la composición de mazos según rareza de cartas.  

**Pregunta de negocio principal:**  
> ¿Cómo cambian los patrones de uso de cartas, win conditions y rarezas a lo largo de la temporada?


## 3. Situación Actual

- Clash Royale tiene un **meta** que cambia con el tiempo (balance de cartas, ajustes en el juego).  
- Los jugadores tienden a concentrarse en **mazos con alta efectividad**.  
- Analizar la evolución del meta permite:  
  - Entender cómo cambia el balance del juego.  
  - Detectar qué cartas son más determinantes en la victoria.  


## 4. Objetivos de Machine Learning / Data Analysis

Aunque el proyecto se centra en **EDA (Exploratory Data Analysis)**, también puede servir como base para un futuro modelo de ML:

- **Target potencial:** victoria (1) / derrota (0).  
- **Features potenciales:** nivel de cartas, rareza, presencia de win condition, diferencia de niveles.  
- **Uso multi-dataset (3 días):**  
  - Comparar distribuciones entre días.  
  - Entrenar con un día y validar con otro para ver si el modelo generaliza.  


## 5. Plan del Proyecto

El proyecto seguirá la metodología CRISP-DM:

1. **Comprensión del negocio** (este notebook).  
2. **Comprensión de los datos**: descripción, exploración y verificación de calidad.  
3. **Preparación de los datos**: limpieza, transformación y creación de features.  
4. **(Opcional más adelante)**: modelado predictivo.  
5. **(Opcional más adelante)**: evaluación de modelos.


## 6. Resumen Ejecutivo

Este proyecto busca **analizar la evolución del meta en Clash Royale** usando datos de partidas en distintos días de una temporada.  
Los resultados esperados son:

- Identificar las cartas más usadas y su evolución temporal.  
- Determinar las win conditions más frecuentes.  
- Analizar la distribución de rarezas en los mazos.  
- Generar una base para futuros modelos predictivos de victoria.


In [12]:
# Cargar salidas del pipeline de business_understanding desde Kedro
# Nota: Esta celda requiere que la celda de configuración inicial se haya ejecutado

try:
    # Verificar que project_root está definido
    if 'project_root' not in globals():
        raise NameError("project_root no está definido. Ejecuta primero la celda de configuración inicial.")
    
    print("🔄 Inicializando Kedro...")
    
    # Importar y configurar Kedro
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project
    
    # Bootstrap del proyecto
    print("  - Bootstrap del proyecto...")
    metadata = bootstrap_project(project_root)
    print(f"  ✓ Proyecto: {metadata.project_name}")
    print(f"  ✓ Paquete: {metadata.package_name}")
    
    # Crear sesión de Kedro - CORREGIDO: usar solo project_path y context manager
    print("  - Creando sesión de Kedro...")
    # Usar context manager para asegurar que la sesión se cierre correctamente
    session = KedroSession.create(project_path=project_root)
    context = session.load_context()
    
    # Obtener el catálogo
    catalog = context.catalog
    
    print("\n✅ Contexto de Kedro cargado exitosamente")
    # El catálogo es un diccionario, podemos obtener las claves
    try:
        catalog_keys = list(catalog._datasets.keys()) if hasattr(catalog, '_datasets') else []
        print(f"📦 Catálogo disponible con {len(catalog_keys)} datasets")
    except:
        print("📦 Catálogo disponible")
    
    # Cargar datos del catálogo
    print("\n📊 Cargando datos del catálogo...")
    
    # Intentar cargar datasets limpios
    datasets_loaded = {}
    for dataset_name in ["Combates1_cleaned", "Combates2_cleaned", "Combates3_cleaned"]:
        try:
            data = catalog.load(dataset_name)
            datasets_loaded[dataset_name] = data
            print(f"✓ {dataset_name}: {data.shape}")
        except Exception as e:
            print(f"⚠ {dataset_name} no disponible: {str(e)[:80]}")
            datasets_loaded[dataset_name] = None
    
    # Asignar a variables
    combates1_cleaned = datasets_loaded.get("Combates1_cleaned")
    combates2_cleaned = datasets_loaded.get("Combates2_cleaned")
    combates3_cleaned = datasets_loaded.get("Combates3_cleaned")

    # Intentar cargar análisis de business understanding
    analysis_data = {}
    for analysis_name in ["business_objectives_analysis", "current_situation_evaluation", 
                         "ml_objectives_definition", "project_plan", "business_understanding_summary"]:
        try:
            data = catalog.load(analysis_name)
            analysis_data[analysis_name] = data
            print(f"✓ {analysis_name} cargado")
        except Exception as e:
            print(f"⚠ {analysis_name} no disponible: {str(e)[:80]}")
            analysis_data[analysis_name] = None
    
    # Asignar a variables
    business_objectives = analysis_data.get("business_objectives_analysis")
    current_situation = analysis_data.get("current_situation_evaluation")
    ml_objectives = analysis_data.get("ml_objectives_definition")
    project_plan = analysis_data.get("project_plan")
    summary = analysis_data.get("business_understanding_summary")
    
    # Mostrar resumen si está disponible
    if summary is not None:
        print("\n📋 Resumen Ejecutivo:")
        try:
            display(summary)
        except:
            print(summary)
    
    print("\n💡 Nota: Si los datos no están disponibles, ejecuta primero:")
    print("   kedro run --pipeline=business_understanding")
    
except NameError as e:
    print(f"\n❌ Error: {e}")
    print("\n💡 Ejecuta primero la celda de configuración inicial")
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\n💡 Posibles soluciones:")
    print("   1. Verifica que ejecutaste la celda de configuración inicial")
    print("   2. Verifica que pyproject.toml existe")
    print("   3. Si los datos no están disponibles, ejecuta: kedro run --pipeline=business_understanding")
    import traceback
    traceback.print_exc()


🔄 Inicializando Kedro...
  - Bootstrap del proyecto...
  ✓ Proyecto: ML ClashRoyale
  ✓ Paquete: proyecto_ml_clashroyale
  - Creando sesión de Kedro...


[11/28/25 15:45:18] WARNING  c:\Users\PC                                                            warnings.py:112
                             RST\Documents\GitHub\ML_ClashRoyale\venv\Lib\site-packages\kedro\frame                
                             work\project\__init__.py:350: UserWarning: The                                        
                             'proyecto_ml_clashroyale.pipelines.nodes' module does not expose a                    
                             'create_pipeline' function, so no pipelines defined therein will be                   
                             returned by 'find_pipelines'.                                                         
                               warnings.warn(                                                                      
                                                                                                                   

                    INFO     Kedro is sending anonymous usage data with the sole purpose of improving plugin.py:243
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/configuration/telemetry.html                         


✅ Contexto de Kedro cargado exitosamente
📦 Catálogo disponible con 131 datasets

📊 Cargando datos del catálogo...


[11/28/25 15:45:19] INFO     Loading data from Combates1_cleaned (CSVDataset)...               data_catalog.py:1048

...

✓ project_plan cargado


                    INFO     Loading data from business_understanding_summary                  data_catalog.py:1048
                             (PickleDataset)...                                                                    

✓ business_understanding_summary cargado

📋 Resumen Ejecutivo:



{
    'resumen_ejecutivo': {
        'proyecto': 'Análisis de Estrategias en Clash Royale',
        'objetivo_principal': 'Analizar estrategias efectivas en Clash Royale',
        'datos_disponibles': '5,644,203 batallas históricas',
        'objetivos_clave': [
            'Identificar las cartas más utilizadas en los mazos',
            'Analizar las win conditions más efectivas',
            'Estudiar la distribución de rarezas en los mazos',
            'Comprender patrones de éxito en las batallas'
        ],
        'valor_esperado': 'Mejorar la comprensión de estrategias efectivas para jugadores y desarrolladores'
    },
    'siguientes_pasos': [
        'Ejecutar pipeline de EDA',
        'Analizar distribuciones de variables clave',
        'Identificar cartas más utilizadas',
        'Evaluar efectividad de win conditions'
    ],
    'recomendaciones': [
        'Enfocar análisis en patrones de uso de cartas',
        'Identificar win conditions más efectivas',
        'Anal


💡 Nota: Si los datos no están disponibles, ejecuta primero:
   kedro run --pipeline=business_understanding
